# C09 — Diffusion Models: DDPM, DDIM, and Stable Diffusion

> **Audience**: PhD students · **Framework**: PyTorch + diffusers · **Dataset**: synthetic sprites

Diffusion models are the dominant generative paradigm for images, audio, and
molecular design. They surpass GANs on image quality while being more stable to train.

**What this notebook builds from scratch**
1. Forward diffusion process — analytically adding noise at any timestep
2. Noise schedule — linear and cosine beta schedules
3. ContextUNet — the conditional denoising backbone
4. DDPM training — predict the noise added to corrupted images
5. DDPM sampling — iterative denoising from pure noise
6. DDIM — deterministic fast sampling (10-50× fewer steps)
7. Stable Diffusion — text-to-image via diffusers (3 lines)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
print(f"Device: {DEVICE}")

# 1) The Forward Diffusion Process

**Intuition**
Diffusion models define a *forward process* that gradually adds Gaussian noise
to a clean image $x_0$ over $T$ timesteps, eventually producing pure noise $x_T \sim \mathcal{N}(0, I)$.

The model learns to *reverse* this process — denoising step by step.

**The key mathematical insight** (Ho et al., 2020 — DDPM)

We can skip the sequential process and sample $x_t$ at any timestep $t$ directly:

$$q(x_t | x_0) = \mathcal{N}(x_t;\; \sqrt{\bar{\alpha}_t}\, x_0,\; (1 - \bar{\alpha}_t)\, I)$$

or equivalently (reparametrisation):

$$x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

where $\bar{\alpha}_t = \prod_{s=1}^{t} (1 - \beta_s)$ is the cumulative noise schedule.

**Why this matters for training**
This closed-form expression means we can sample noisy images at arbitrary $t$
*without* running the sequential Markov chain — training is $O(1)$ per sample.

In [ ]:
def linear_beta_schedule(T: int, beta_start: float = 1e-4, beta_end: float = 0.02) -> torch.Tensor:
    """
    Linear noise schedule from Ho et al. (2020).

    Linearly interpolates beta from beta_start (almost no noise at t=1)
    to beta_end (significant noise at t=T).

    Args:
        T          : Total diffusion timesteps
        beta_start : Starting noise level
        beta_end   : Ending noise level

    Returns:
        betas : (T,) noise schedule
    """
    # (T,)
    return torch.linspace(beta_start, beta_end, T)


def cosine_beta_schedule(T: int, s: float = 0.008) -> torch.Tensor:
    """
    Cosine noise schedule from Nichol & Dhariwal (2021).

    Avoids the linear schedule's abrupt increase near t=T, giving
    smoother training and better image quality at the same T.

    Args:
        T : Total diffusion timesteps
        s : Offset to prevent betas from being too small near t=0

    Returns:
        betas : (T,) noise schedule
    """
    steps    = T + 1
    x        = torch.linspace(0, T, steps)
    # Cosine alpha_bar schedule
    # (T+1,)
    alpha_bar = torch.cos(((x / T) + s) / (1 + s) * torch.pi / 2) ** 2
    alpha_bar = alpha_bar / alpha_bar[0]  # normalise to start at 1
    # (T,)
    betas     = 1 - (alpha_bar[1:] / alpha_bar[:-1])
    return betas.clamp(min=1e-5, max=0.999)


class DiffusionSchedule:
    """
    Pre-computes all quantities needed for forward diffusion and sampling.

    Stores:
        betas        : (T,) noise levels
        alphas       : (T,) = 1 - betas
        alpha_bar    : (T,) cumulative product of alphas
        sqrt_ab      : (T,) sqrt(alpha_bar)     — signal coefficient
        sqrt_1m_ab   : (T,) sqrt(1-alpha_bar)   — noise coefficient
        posterior_var: (T,) variance for DDPM reverse step
    """

    def __init__(self, T: int = 400, schedule: str = "cosine") -> None:
        self.T = T

        betas = cosine_beta_schedule(T) if schedule == "cosine" else linear_beta_schedule(T)

        alphas   = 1.0 - betas                           # (T,)
        alpha_bar = torch.cumprod(alphas, dim=0)         # (T,)

        # Quantities needed for forward process q(x_t | x_0)
        self.sqrt_ab    = alpha_bar.sqrt()               # (T,)
        self.sqrt_1m_ab = (1 - alpha_bar).sqrt()         # (T,)

        # Quantities needed for DDPM reverse step
        alpha_bar_prev  = F.pad(alpha_bar[:-1], (1, 0), value=1.0)  # (T,)
        self.posterior_var = betas * (1 - alpha_bar_prev) / (1 - alpha_bar)  # (T,)
        self.betas          = betas
        self.alphas         = alphas
        self.alpha_bar      = alpha_bar

    def _broadcast(self, coeff: torch.Tensor, t: torch.Tensor, shape: tuple) -> torch.Tensor:
        """Gathers schedule values at timestep t and broadcasts to input shape."""
        # (batch_num,) → (batch_num, 1, 1, 1) for NCHW broadcasting
        vals = coeff[t]
        return vals.view(t.shape[0], *([1] * (len(shape) - 1)))

    def q_sample(
        self, x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor = None
    ) -> tuple:
        """
        Samples x_t given x_0 and timestep t using the closed-form reparametrisation.

        x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * epsilon

        Args:
            x0    : (batch_num, n_channels, H, W) clean images
            t     : (batch_num,) integer timesteps
            noise : Optional pre-generated noise; sampled if None

        Returns:
            x_t   : (batch_num, n_channels, H, W) noisy images at timestep t
            noise : (batch_num, n_channels, H, W) the noise that was added
        """
        if noise is None:
            noise = torch.randn_like(x0)

        # Signal and noise coefficients, broadcast to image shape
        sqrt_ab   = self._broadcast(self.sqrt_ab,    t, x0.shape).to(x0.device)
        sqrt_1mab = self._broadcast(self.sqrt_1m_ab, t, x0.shape).to(x0.device)

        # Closed-form q(x_t | x_0)
        # (batch_num, n_channels, H, W)
        x_t = sqrt_ab * x0 + sqrt_1mab * noise
        return x_t, noise


# ── Visualise forward diffusion ────────────────────────────────────────────────
schedule = DiffusionSchedule(T=400, schedule="cosine")

x0_demo = torch.randn(1, 1, 16, 16)  # fake 16×16 grayscale image
timesteps = [0, 50, 100, 200, 300, 400]

fig, axes = plt.subplots(1, len(timesteps), figsize=(14, 2))
for ax, t_val in zip(axes, timesteps):
    t_val = min(t_val, 399)
    t_tensor = torch.tensor([t_val])
    x_t, _   = schedule.q_sample(x0_demo, t_tensor)
    ax.imshow(x_t[0, 0].numpy(), cmap="gray", vmin=-3, vmax=3)
    ax.set_title(f"t={t_val}", fontsize=8); ax.axis("off")
plt.suptitle("Forward diffusion: clean → noise", y=1.05)
plt.tight_layout(); plt.show()

# Visualise alpha_bar to understand noise schedule
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.plot(schedule.alpha_bar.numpy()); ax1.set_title("alpha_bar (signal fraction)"); ax1.set_xlabel("t")
ax2.plot(schedule.betas.numpy());     ax2.set_title("betas (noise increment)");     ax2.set_xlabel("t")
plt.tight_layout(); plt.show()

# 2) ContextUNet — Conditional Denoising Backbone

**The denoising objective**
Given $x_t$ (noisy image) and timestep $t$, predict the noise $\epsilon$:

$$L_{simple} = \mathbb{E}_{x_0, t, \epsilon}\left[ \| \epsilon - \epsilon_\theta(x_t, t) \|^2 \right]$$

This is a simple mean squared error between the actual noise added and the
predicted noise. The *predicting noise* (Ho et al.) formulation is more stable
than predicting $x_0$ directly.

**Architecture: ContextUNet**
A U-Net conditioned on:
1. **Timestep $t$**: sinusoidal embedding + MLP → added to every residual block
2. **Context $c$** (optional): class label or text embedding → added via FiLM (Feature-wise Linear Modulation)

The sinusoidal timestep embedding is identical to the positional encoding
in the original Transformer — it allows the network to distinguish noise levels.

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    """
    Sinusoidal timestep embedding — identical to Transformer positional encoding.

    Maps integer timestep t → a fixed-frequency sinusoidal vector,
    then projects it to embed_dim via a small MLP.

    This gives the denoiser information about the current noise level
    without requiring an additional input channel.
    """

    def __init__(self, embed_dim: int) -> None:
        super().__init__()

        assert embed_dim % 2 == 0

        # MLP to project sinusoidal features → embed_dim
        # (batch_num, embed_dim) → (batch_num, embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.SiLU(),  # Swish activation — standard in diffusion models
            nn.Linear(embed_dim * 4, embed_dim),
        )
        self.embed_dim = embed_dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """
        Args:
            t : (batch_num,) integer timesteps

        Returns:
            emb : (batch_num, embed_dim)
        """
        half_dim = self.embed_dim // 2
        # Frequencies: 1/10000^(2i/d) for i = 0..half_dim-1
        freqs = torch.exp(
            -torch.arange(half_dim, device=t.device) * (torch.log(torch.tensor(10000.0)) / half_dim)
        )
        # Outer product: (batch_num, half_dim)
        args = t.float()[:, None] * freqs[None, :]

        # Sine and cosine features concatenated
        # (batch_num, embed_dim)
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

        return self.mlp(emb)


class ResBlock(nn.Module):
    """
    Residual block for ContextUNet with timestep and context conditioning.

    Applies FiLM (Feature-wise Linear Modulation): the time embedding scales
    and shifts the feature maps, allowing the network to modulate its behaviour
    based on the current noise level.
    """

    def __init__(self, in_channels: int, out_channels: int, time_dim: int, context_dim: int = 0) -> None:
        super().__init__()

        # Main conv path
        # (batch_num, in_channels, H, W) → (batch_num, out_channels, H, W)
        self.conv = nn.Sequential(
            nn.GroupNorm(8, in_channels),
            nn.SiLU(),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        )

        # Time scale+shift via FiLM: project time embedding to 2×out_channels
        # (batch_num, time_dim) → (batch_num, 2*out_channels)
        self.time_proj = nn.Linear(time_dim, 2 * out_channels)

        # Optional context conditioning
        self.context_dim = context_dim
        if context_dim > 0:
            self.ctx_proj = nn.Linear(context_dim, out_channels)

        # 1×1 conv to match channels when they change
        self.shortcut = (
            nn.Conv2d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor, ctx: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            x     : (batch_num, in_channels, H, W)
            t_emb : (batch_num, time_dim) timestep embedding
            ctx   : (batch_num, context_dim) optional context (class / text)

        Returns:
            out   : (batch_num, out_channels, H, W)
        """
        # Main feature path
        # (batch_num, in_channels, H, W) → (batch_num, out_channels, H, W)
        h = self.conv(x)

        # FiLM conditioning: time embedding produces scale (γ) and shift (β)
        # (batch_num, time_dim) → (batch_num, 2*out_channels)
        time_scale_shift = self.time_proj(t_emb)[:, :, None, None]  # broadcast to H,W
        scale, shift     = time_scale_shift.chunk(2, dim=1)          # each (batch_num, out_channels, 1, 1)
        h = h * (1 + scale) + shift

        # Optional context injection
        if ctx is not None and self.context_dim > 0:
            # (batch_num, context_dim) → (batch_num, out_channels, 1, 1)
            ctx_emb = self.ctx_proj(ctx)[:, :, None, None]
            h = h + ctx_emb

        # Residual connection (with 1×1 projection if channel counts differ)
        return h + self.shortcut(x)


class ContextUNet(nn.Module):
    """
    U-Net denoiser conditioned on timestep and optional context (class/text).

    Architecture:
        Stem → [Encoder: ResBlock×2 + Downsample] × 3
             → Bottleneck: ResBlock×2
             → [Decoder: Upsample + ResBlock×2] × 3
             → Output Conv
    """

    def __init__(
        self,
        in_channels: int  = 1,
        model_dim: int    = 64,
        time_dim: int     = 256,
        context_dim: int  = 0,
        num_classes: int  = 0,
    ) -> None:
        super().__init__()

        # Timestep embedding
        self.time_embed = SinusoidalTimeEmbedding(time_dim)

        # Class label embedding (optional)
        self.has_context = (context_dim > 0 or num_classes > 0)
        self.context_dim = context_dim if context_dim > 0 else time_dim
        if num_classes > 0:
            self.class_embed = nn.Embedding(num_classes, self.context_dim)

        c = model_dim

        # Stem
        self.stem = nn.Conv2d(in_channels, c, kernel_size=3, padding=1)

        # Encoder
        self.enc1 = ResBlock(c,     c * 2, time_dim, self.context_dim)
        self.enc2 = ResBlock(c * 2, c * 4, time_dim, self.context_dim)
        self.enc3 = ResBlock(c * 4, c * 8, time_dim, self.context_dim)
        self.down  = nn.AvgPool2d(2)

        # Bottleneck
        self.bot1  = ResBlock(c * 8, c * 8, time_dim, self.context_dim)
        self.bot2  = ResBlock(c * 8, c * 8, time_dim, self.context_dim)

        # Decoder (skip connections double the input channels)
        self.dec3  = ResBlock(c * 8 + c * 8, c * 4, time_dim, self.context_dim)
        self.dec2  = ResBlock(c * 4 + c * 4, c * 2, time_dim, self.context_dim)
        self.dec1  = ResBlock(c * 2 + c * 2, c,     time_dim, self.context_dim)

        # Final output: map back to input channel count
        self.out_conv = nn.Sequential(
            nn.GroupNorm(8, c),
            nn.SiLU(),
            nn.Conv2d(c, in_channels, kernel_size=1),
        )

    def forward(
        self,
        x: torch.Tensor,
        t: torch.Tensor,
        ctx: torch.Tensor = None,
    ) -> torch.Tensor:
        """
        Args:
            x   : (batch_num, in_channels, H, W) noisy image at timestep t
            t   : (batch_num,) integer timesteps
            ctx : (batch_num,) class indices or (batch_num, context_dim) context vectors

        Returns:
            noise_pred : (batch_num, in_channels, H, W) predicted noise
        """
        # Embed timestep → continuous vector
        # (batch_num,) → (batch_num, time_dim)
        t_emb = self.time_embed(t)

        # Embed class labels if provided
        if ctx is not None and hasattr(self, "class_embed"):
            ctx_emb = self.class_embed(ctx)  # (batch_num, context_dim)
        else:
            ctx_emb = None

        # Stem
        # (batch_num, in_channels, H, W) → (batch_num, model_dim, H, W)
        x0 = self.stem(x)

        # Encoder + save skip connections
        s1 = self.enc1(x0,          t_emb, ctx_emb)  # (batch_num, 2c, H, W)
        s2 = self.enc2(self.down(s1), t_emb, ctx_emb)  # (batch_num, 4c, H/2, W/2)
        s3 = self.enc3(self.down(s2), t_emb, ctx_emb)  # (batch_num, 8c, H/4, W/4)

        # Bottleneck
        b  = self.bot1(self.down(s3), t_emb, ctx_emb)  # (batch_num, 8c, H/8, W/8)
        b  = self.bot2(b,              t_emb, ctx_emb)  # (batch_num, 8c, H/8, W/8)

        # Decoder with skip connections
        # (batch_num, 8c+8c, H/4, W/4) → (batch_num, 4c, H/4, W/4)
        d3 = self.dec3(torch.cat([F.interpolate(b,  size=s3.shape[2:], mode="nearest"), s3], dim=1), t_emb, ctx_emb)
        # (batch_num, 4c+4c, H/2, W/2) → (batch_num, 2c, H/2, W/2)
        d2 = self.dec2(torch.cat([F.interpolate(d3, size=s2.shape[2:], mode="nearest"), s2], dim=1), t_emb, ctx_emb)
        # (batch_num, 2c+2c, H, W) → (batch_num, c, H, W)
        d1 = self.dec1(torch.cat([F.interpolate(d2, size=s1.shape[2:], mode="nearest"), s1], dim=1), t_emb, ctx_emb)

        # (batch_num, c, H, W) → (batch_num, in_channels, H, W)
        return self.out_conv(d1)


# ── Dry-run ────────────────────────────────────────────────────────────────────
unet_ddpm = ContextUNet(in_channels=1, model_dim=32, time_dim=128, num_classes=5)
x_noise   = torch.zeros(4, 1, 16, 16)
t_test    = torch.randint(0, 400, (4,))
ctx_test  = torch.randint(0, 5, (4,))  # class labels as context

with torch.no_grad():
    pred_noise = unet_ddpm(x_noise, t_test, ctx_test)

print(f"Input  x_t         : {x_noise.shape}")     # (4, 1, 16, 16)
print(f"Predicted noise    : {pred_noise.shape}")   # (4, 1, 16, 16)  ← same shape
print(f"Parameters         : {sum(p.numel() for p in unet_ddpm.parameters()):,}")

# 3) DDPM Training

The training objective is deceptively simple:

```
For each batch:
  1. Sample random timesteps t ~ Uniform(1, T)
  2. Sample noise eps ~ N(0, I)
  3. Corrupt images: x_t = sqrt(alpha_bar_t) * x0 + sqrt(1-alpha_bar_t) * eps
  4. Predict noise: eps_hat = model(x_t, t, context)
  5. Loss = MSE(eps_hat, eps)
```

**Why MSE on noise rather than on x_0?**
Ho et al. showed empirically that predicting $\epsilon$ rather than $x_0$
gives better sample quality. Theoretically, predicting $\epsilon$ corresponds
to a specific weighting of the ELBO that emphasises high-noise timesteps.

In [ ]:
class SpriteDataset(Dataset):
    """
    Synthetic sprite dataset: 16×16 greyscale images of simple shapes.
    Each class is a distinct geometric pattern (circle, cross, star, rect, diamond).
    """

    def __init__(self, num_samples: int = 5000, img_size: int = 16, num_classes: int = 5) -> None:
        np.random.seed(1)
        self.imgs   = []
        self.labels = []
        patterns    = [self._circle, self._cross, self._star, self._rect, self._diamond]

        for _ in range(num_samples):
            cls = np.random.randint(num_classes)
            img = np.zeros((img_size, img_size), dtype=np.float32)
            patterns[cls % len(patterns)](img, img_size)
            img += np.random.randn(*img.shape) * 0.05  # slight noise
            self.imgs.append(torch.tensor(img).unsqueeze(0))  # (1, H, W)
            self.labels.append(cls)

    def _circle(self, img, s):
        Y, X = np.ogrid[:s, :s]; c=s//2; r=s//3
        img[(X-c)**2 + (Y-c)**2 <= r**2] = 1.0
    def _cross(self, img, s):
        c=s//2; w=s//6
        img[c-w:c+w, :] = 1.0; img[:, c-w:c+w] = 1.0
    def _star(self, img, s):
        c=s//2; r=s//3
        for angle in np.linspace(0, np.pi, 5):
            x=int(c+r*np.cos(angle)); y=int(c+r*np.sin(angle))
            img[max(0,y-1):y+2, max(0,x-1):x+2] = 1.0
    def _rect(self, img, s):
        m=s//4; img[m:-m, m:-m] = 1.0; img[m+2:-m-2, m+2:-m-2] = 0.0
    def _diamond(self, img, s):
        c=s//2; r=s//3
        Y, X = np.ogrid[:s, :s]
        img[np.abs(X-c)+np.abs(Y-c) <= r] = 1.0

    def __len__(self): return len(self.imgs)
    def __getitem__(self, i): return self.imgs[i] * 2 - 1, self.labels[i]  # scale to [-1, 1]


sprite_ds     = SpriteDataset(num_samples=5000, img_size=16, num_classes=5)
sprite_loader = DataLoader(sprite_ds, batch_size=128, shuffle=True, drop_last=True)

# Visualise dataset
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
class_names = ["circle", "cross", "star", "rect", "diamond"]
for row in range(2):
    for col in range(5):
        img, lbl = sprite_ds[row * 5 + col]
        axes[row, col].imshow(img[0].numpy(), cmap="gray", vmin=-1, vmax=1)
        axes[row, col].set_title(class_names[lbl], fontsize=8); axes[row, col].axis("off")
plt.suptitle("Synthetic sprite dataset (16×16)")
plt.tight_layout(); plt.show()


def train_ddpm(model, loader, schedule, n_epochs=50, lr=2e-4):
    """
    DDPM training: sample t, corrupt x0, predict noise, compute MSE.

    Args:
        model    : ContextUNet
        loader   : DataLoader of (x0, context_labels)
        schedule : DiffusionSchedule
        n_epochs : Training epochs

    Returns:
        loss_history : List of mean losses per epoch
    """
    model.to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    history   = []

    for epoch in range(1, n_epochs + 1):
        model.train()
        running_loss = 0.0

        for x0, ctx in loader:
            x0, ctx = x0.to(DEVICE), ctx.to(DEVICE)

            # Sample uniformly random timesteps for each image in batch
            # (batch_num,)
            t     = torch.randint(0, schedule.T, (x0.size(0),), device=DEVICE)

            # Generate random noise and corrupt x0 at timestep t
            # (batch_num, 1, H, W)
            noise = torch.randn_like(x0)
            x_t, noise = schedule.q_sample(x0, t.cpu(), noise)
            x_t   = x_t.to(DEVICE)

            optimizer.zero_grad()

            # Predict the noise that was added
            # (batch_num, 1, H, W) → (batch_num, 1, H, W)
            noise_pred = model(x_t, t, ctx)

            # Simple MSE between predicted and actual noise
            loss = F.mse_loss(noise_pred, noise.to(DEVICE))
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * x0.size(0)

        scheduler.step()
        mean_loss = running_loss / len(loader.dataset)
        history.append(mean_loss)

        if epoch % 10 == 0 or epoch == 1:
            print(f"Epoch {epoch:03d}/{n_epochs} | loss={mean_loss:.5f}")

    return history


torch.manual_seed(0)
ddpm_model = ContextUNet(in_channels=1, model_dim=32, time_dim=128, num_classes=5)
print(f"DDPM UNet parameters: {sum(p.numel() for p in ddpm_model.parameters()):,}")
history_ddpm = train_ddpm(ddpm_model, sprite_loader, schedule, n_epochs=50, lr=2e-4)

plt.figure(figsize=(7, 3))
plt.plot(history_ddpm); plt.xlabel("Epoch"); plt.ylabel("MSE loss")
plt.title("DDPM training loss"); plt.tight_layout(); plt.show()

# 4) DDPM Sampling

**Reverse process** (iterative denoising)

Starting from pure noise $x_T \sim \mathcal{N}(0, I)$, we iteratively apply:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{1 - \alpha_t}{\sqrt{1 - \bar{\alpha}_t}} \epsilon_\theta(x_t, t) \right) + \sigma_t z, \quad z \sim \mathcal{N}(0, I)$$

This requires **T forward passes** through the model — the main computational
bottleneck of DDPM. DDIM (Section 5) addresses this.

In [ ]:
@torch.no_grad()
def ddpm_sample(
    model: nn.Module,
    schedule: DiffusionSchedule,
    n_samples: int,
    img_shape: tuple,
    ctx: torch.Tensor = None,
    device: torch.device = DEVICE,
) -> torch.Tensor:
    """
    Generates images via iterative DDPM reverse process.

    Runs T model forward passes to go from pure noise to samples.

    Args:
        model     : Trained ContextUNet
        schedule  : DiffusionSchedule with precomputed noise schedule
        n_samples : Number of images to generate
        img_shape : (n_channels, H, W) shape of each image
        ctx       : (n_samples,) optional conditioning labels
        device    : Computation device

    Returns:
        x0 : (n_samples, *img_shape) generated images in [-1, 1]
    """
    model.eval()

    # Start from pure Gaussian noise at t=T
    # (n_samples, n_channels, H, W)
    x_t = torch.randn(n_samples, *img_shape, device=device)

    for t_val in reversed(range(schedule.T)):
        # Batch timestep tensor
        # (n_samples,)
        t_tensor = torch.full((n_samples,), t_val, dtype=torch.long, device=device)

        # Predict noise at this timestep
        # (n_samples, n_channels, H, W) → (n_samples, n_channels, H, W)
        noise_pred = model(x_t, t_tensor, ctx)

        # Retrieve schedule quantities
        alpha_t   = schedule.alphas[t_val].to(device)
        alpha_bar = schedule.alpha_bar[t_val].to(device)
        beta_t    = schedule.betas[t_val].to(device)

        # Compute x_{t-1} mean (DDPM reverse step)
        # (n_samples, n_channels, H, W)
        x_t_prev_mean = (1 / alpha_t.sqrt()) * (
            x_t - (beta_t / (1 - alpha_bar).sqrt()) * noise_pred
        )

        # Add stochastic noise at all steps except the last (t=0)
        if t_val > 0:
            sigma_t = schedule.posterior_var[t_val].sqrt().to(device)
            x_t = x_t_prev_mean + sigma_t * torch.randn_like(x_t)
        else:
            x_t = x_t_prev_mean

    return x_t


# Generate 10 samples conditioned on each of the 5 sprite classes
ctx_labels = torch.arange(5, device=DEVICE).repeat(2)  # 2 samples per class
samples_ddpm = ddpm_sample(ddpm_model, schedule, n_samples=10,
                            img_shape=(1, 16, 16), ctx=ctx_labels)

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(samples_ddpm[i, 0].cpu().numpy(), cmap="gray", vmin=-1, vmax=1)
    ax.set_title(class_names[ctx_labels[i].item()], fontsize=8); ax.axis("off")
plt.suptitle("DDPM samples (conditioned on class)")
plt.tight_layout(); plt.show()

# 5) DDIM — Deterministic Fast Sampling

**The problem with DDPM sampling**
DDPM requires T=400 (or 1000) network forward passes to generate one image.
On modern hardware this takes seconds per image.

**DDIM** (Song et al., 2020) reformulates the generative process as an ODE
instead of an SDE, enabling:
1. **Deterministic sampling**: same latent $x_T$ always produces the same image
2. **Large step skipping**: we can skip most timesteps and still get good quality
3. **10–50× speedup**: sample with 20–50 steps instead of 1000

**DDIM update rule** (non-Markovian, deterministic when $\eta=0$)

$$x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} \cdot \hat{x}_0 + \sqrt{1-\bar{\alpha}_{t-1} - \sigma_t^2} \cdot \epsilon_\theta + \sigma_t z$$

where $\hat{x}_0 = (x_t - \sqrt{1-\bar{\alpha}_t} \cdot \epsilon_\theta) / \sqrt{\bar{\alpha}_t}$
is the *predicted clean image* at each step, and $\sigma_t = 0$ for deterministic sampling.

In [ ]:
@torch.no_grad()
def ddim_sample(
    model: nn.Module,
    schedule: DiffusionSchedule,
    n_samples: int,
    img_shape: tuple,
    ctx: torch.Tensor  = None,
    n_steps: int       = 50,
    eta: float         = 0.0,
    device: torch.device = DEVICE,
) -> torch.Tensor:
    """
    Generates images with DDIM — deterministic when eta=0.

    Args:
        model     : Trained ContextUNet (same model as DDPM, no retraining needed)
        schedule  : DiffusionSchedule
        n_samples : Number of images to generate
        img_shape : (n_channels, H, W)
        ctx       : (n_samples,) optional conditioning labels
        n_steps   : Number of denoising steps (much fewer than T)
        eta       : Stochasticity parameter: 0=deterministic DDIM, 1=DDPM

    Returns:
        x0 : (n_samples, *img_shape) generated images in [-1, 1]
    """
    model.eval()

    # Subsampled timestep sequence: pick n_steps evenly from [0, T]
    step_size   = schedule.T // n_steps
    timesteps   = list(reversed(range(0, schedule.T, step_size)))  # T → 0

    # Start from pure noise
    # (n_samples, n_channels, H, W)
    x_t = torch.randn(n_samples, *img_shape, device=device)

    for i, t_curr in enumerate(timesteps):
        t_prev = timesteps[i + 1] if i + 1 < len(timesteps) else 0

        t_tensor = torch.full((n_samples,), t_curr, dtype=torch.long, device=device)

        # Predict noise
        # (n_samples, n_channels, H, W)
        noise_pred = model(x_t, t_tensor, ctx)

        # Retrieve schedule quantities
        ab_curr = schedule.alpha_bar[t_curr].to(device)
        ab_prev = schedule.alpha_bar[t_prev].to(device)

        # Reconstruct x0 estimate from current x_t and predicted noise
        # x0_hat = (x_t - sqrt(1-ab) * eps) / sqrt(ab)
        # (n_samples, n_channels, H, W)
        x0_hat = (x_t - (1 - ab_curr).sqrt() * noise_pred) / ab_curr.sqrt()

        # DDIM stochasticity — sigma=0 for deterministic, sigma>0 for stochastic
        sigma_t = eta * ((1 - ab_prev) / (1 - ab_curr)).sqrt() * (1 - ab_curr / ab_prev).sqrt()

        # Direction toward x_t (the "predicted direction" in DDIM notation)
        # (n_samples, n_channels, H, W)
        dir_xt = (1 - ab_prev - sigma_t**2).sqrt() * noise_pred

        # DDIM update step
        # (n_samples, n_channels, H, W)
        x_t = ab_prev.sqrt() * x0_hat + dir_xt
        if sigma_t > 0:
            x_t = x_t + sigma_t * torch.randn_like(x_t)

    return x_t


# Compare DDPM (400 steps) vs DDIM (20 steps) for the same latent
import time

torch.manual_seed(123)
ctx_fixed = torch.arange(5, device=DEVICE).repeat(2)

t0 = time.time()
samples_ddim = ddim_sample(ddpm_model, schedule, n_samples=10, img_shape=(1,16,16),
                            ctx=ctx_fixed, n_steps=20, eta=0.0)
t_ddim = time.time() - t0

torch.manual_seed(123)
torch.randn(10, 1, 16, 16)  # consume same seed state for fair comparison
t0 = time.time()
samples_ddpm2 = ddpm_sample(ddpm_model, schedule, n_samples=10, img_shape=(1,16,16), ctx=ctx_fixed)
t_ddpm = time.time() - t0

print(f"DDPM (T=400) time : {t_ddpm:.2f}s")
print(f"DDIM (n=20)  time : {t_ddim:.2f}s")
print(f"Speedup            : {t_ddpm/t_ddim:.1f}x")

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes[0]):
    ax.imshow(samples_ddpm2[i, 0].cpu().numpy(), cmap="gray", vmin=-1, vmax=1); ax.axis("off")
for i, ax in enumerate(axes[1]):
    ax.imshow(samples_ddim[i, 0].cpu().numpy(), cmap="gray", vmin=-1, vmax=1); ax.axis("off")
axes[0, 0].set_ylabel("DDPM
(400 steps)", fontsize=9)
axes[1, 0].set_ylabel("DDIM
(20 steps)",  fontsize=9)
plt.suptitle("DDPM vs DDIM quality comparison (same model)"); plt.tight_layout(); plt.show()

# 6) Stable Diffusion via Diffusers

Stable Diffusion extends DDPM with three key innovations:

1. **Latent diffusion**: operates in the *compressed latent space* of a VAE (4×64×64)
   rather than pixel space (3×512×512). This is the main compute reduction.
2. **CLIP text encoder**: conditions the denoiser on text embeddings via cross-attention
3. **Classifier-free guidance (CFG)**: interpolates between conditioned and unconditioned
   predictions, sharpening adherence to the text prompt at the cost of diversity

$$\tilde{\epsilon}_\theta(x_t, c) = \epsilon_\theta(x_t, \varnothing) + w \cdot (\epsilon_\theta(x_t, c) - \epsilon_\theta(x_t, \varnothing))$$

where $w$ is the guidance scale and $c$ is the text condition.

In [ ]:
# Install: !pip install diffusers accelerate --quiet

from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import torch

# Load SD 2.1-base (requires ~5GB VRAM or can run on CPU slowly)
# model_id  = "stabilityai/stable-diffusion-2-1-base"
# pipe = StableDiffusionPipeline.from_pretrained(
#     model_id,
#     torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
# ).to(DEVICE)
#
# Swap to DPM-Solver++ (20 steps vs default 50 DDIM steps)
# pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
#
# ── Generate an image ──────────────────────────────────────────────────────────
# image = pipe(
#     prompt         = "A vibrant oil painting of a red panda coding at a laptop, 4K detailed",
#     negative_prompt= "blurry, low quality, watermark",
#     num_inference_steps = 20,
#     guidance_scale = 7.5,         # w in CFG equation — higher = more prompt-aligned
#     generator      = torch.manual_seed(42),
# ).images[0]
#
# image.save("sd_output.png")
# plt.figure(figsize=(5,5)); plt.imshow(image); plt.axis("off"); plt.show()

print("Stable Diffusion pseudocode shown.")
print("Install diffusers + accelerate and uncomment to run (requires ~5GB VRAM).")
print()
print("Key architectural connections to our DDPM implementation:")
print("  forward diffusion  → same q(x_t|x_0) maths, but in latent space")
print("  ContextUNet        → replaced by a larger UNet with cross-attention to CLIP text")
print("  DDPM sampling      → replaced by DPM-Solver++ (~20 NFE)")
print("  class conditioning → replaced by text conditioning via cross-attention")

# Summary

| Component | DDPM | DDIM | Stable Diffusion |
|---|---|---|---|
| Noise space | Pixel | Pixel | Latent (4×H/8×W/8) |
| Sampling steps | T (400-1000) | n_steps (20-50) | n_steps (20-50) |
| Deterministic | No (SDE) | Yes (ODE, η=0) | Yes (with fixed seed) |
| Conditioning | Class labels | Class labels | CLIP text |
| Guidance | None | None | CFG |

**The key equations to internalise**

Forward (closed-form):
$$x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon$$

Training objective:
$$L = \mathbb{E}\left[\| \epsilon - \epsilon_\theta(x_t, t) \|^2\right]$$

DDPM reverse:
$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\epsilon_\theta\right) + \sigma_t z$$

**Research directions**
- **Consistency models** (Song et al., 2023): distil diffusion into a single-step model
- **Flow matching** (Lipman et al., 2022): ODE-based alternative with simpler training
- **DiT** (Peebles & Xie, 2022): transformer backbone replacing U-Net in latent diffusion